In [ ]:
#to check gpu
import torch

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))


2.11.0+cu128
True
Tesla T4


In [ ]:
!pip install torch torchvision torchaudio
!pip install opencv-python
!pip install sympy
!pip install pandas
!pip install matplotlib
!pip install pillow
!pip install albumentations
!pip install tqdm

In [ ]:
import os

folders = [
    "dataset",
    "dataset/train",
    "dataset/validation",
    "dataset/test",
    "saved_models",
    "results",
    "preprocessing",
    "models",
    "training",
    "inference",
    "verification"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Folders created successfully!")

Folders created successfully!


In [ ]:
import os
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%cd /content/drive/MyDrive/HME100K
!unzip train.zip -d train
!unzip test.zip -d test

Streaming output truncated to the last 5000 lines.
  inflating: test/test_images/test_4533.jpg  
  inflating: test/test_images/test_4534.jpg  
  inflating: test/test_images/test_4535.jpg  
  inflating: test/test_images/test_4536.jpg  
  inflating: test/test_images/test_4538.jpg  
  inflating: test/test_images/test_4539.jpg  
  inflating: test/test_images/test_454.jpg  
  inflating: test/test_images/test_4540.jpg  
  inflating: test/test_images/test_4541.jpg  
  inflating: test/test_images/test_4542.jpg  
  inflating: test/test_images/test_4543.jpg  
  inflating: test/test_images/test_4544.jpg  
  inflating: test/test_images/test_4545.jpg  
  inflating: test/test_images/test_4546.jpg  
  inflating: test/test_images/test_4547.jpg  
  inflating: test/test_images/test_455.jpg  
  inflating: test/test_images/test_4550.jpg  
  inflating: test/test_images/test_4551.jpg  
  inflating: test/test_images/test_4552.jpg  
  inflating: test/test_images/test_4553.jpg  
  inflating: test/test_images/t

In [ ]:
#to check lebels
import os

root = "/content/drive/MyDrive/HME100K"

for path, dirs, files in os.walk(root):
    print("=" * 80)
    print("Folder:", path)
    print("Subfolders:", dirs)
    print("Number of files:", len(files))
    print("First 10 files:", files[:10])

Folder: /content/drive/MyDrive/HME100K
Subfolders: ['subset', 'train', 'test']
Number of files: 2
First 10 files: ['test.zip', 'train.zip']
Folder: /content/drive/MyDrive/HME100K/subset
Subfolders: []
Number of files: 3
First 10 files: ['easy.json', 'hard.json', 'medium.json']
Folder: /content/drive/MyDrive/HME100K/train
Subfolders: ['train_images']
Number of files: 1
First 10 files: ['train_labels.txt']
Folder: /content/drive/MyDrive/HME100K/test
Subfolders: ['test_images']
Number of files: 1
First 10 files: ['test_labels.txt']


In [ ]:
#inspecting label files
label_file = "/content/drive/MyDrive/HME100K/train/train_labels.txt"

with open(label_file, "r", encoding="utf-8") as f:
    for i in range(10):
        print(f.readline())

train_0.jpg	( 2 ) 2 N a O H + C u S O _ { 4 } = N a _ { 2 } S O _ { 4 } + C u ( O H ) _ { 2 } \downarrow

train_2.jpg	\angle A C B = \angle A ^ { \prime } C B ^ { \prime }

train_3.jpg	1 0 \times ( x + 5 ) = 1 3 \times ( \frac { 5 } { 7 } x + 5 )

train_4.jpg	l _ { 2 } = O B = \frac { O A } { 4 } \times 3 = \frac { 1 . 2 m } { 4 } \times 3 = 0 . 9 m

train_6.jpg	\angle A = 4 0 ^ { \circ }

train_8.jpg	2 n + H _ { 2 } S o _ { 4 } = 2 n S _ { 4 } + H _ { 2 } \uparrow

train_9.jpg	\angle B E C = 3 \angle A B E

train_10.jpg	- \frac { b } { 2 0 } = - \frac { - k } { 2 } = \frac { k } { 2 } = 1

train_11.jpg	N a C l O _ { 3 } + K C l = K C l O _ { 3 } + N a C l

train_12.jpg	\textcircled { 1 } \frac { 1 } { 4 } = \frac { 1 \times 2 } { 4 \times 2 } = \frac { 2 } { 8 }



In [ ]:
#read the label file
import pandas as pd

label_file = "/content/drive/MyDrive/HME100K/train/train_labels.txt"

df = pd.read_csv(
    label_file,
    sep="\t",
    header=None,
    names=["image", "label"]
)

print(df.head())

         image                                              label
0  train_0.jpg  ( 2 ) 2 N a O H + C u S O _ { 4 } = N a _ { 2 ...
1  train_2.jpg  \angle A C B = \angle A ^ { \prime } C B ^ { \...
2  train_3.jpg  1 0 \times ( x + 5 ) = 1 3 \times ( \frac { 5 ...
3  train_4.jpg  l _ { 2 } = O B = \frac { O A } { 4 } \times 3...
4  train_6.jpg                         \angle A = 4 0 ^ { \circ }


In [ ]:
#creating vocabulary
from collections import Counter

counter = Counter()

for label in df["label"]:
    counter.update(label)

vocab = sorted(counter.keys())

print("Vocabulary Size:", len(vocab))

print(vocab)


#create dictonary
char2idx = {}

char2idx["<PAD>"] = 0
char2idx["<SOS>"] = 1
char2idx["<EOS>"] = 2
char2idx["<UNK>"] = 3

index = 4

for c in vocab:
    char2idx[c] = index
    index += 1

idx2char = {v:k for k,v in char2idx.items()}

print("Total Tokens:", len(char2idx))

#encoding function
def encode(text):

    encoded = []

    encoded.append(char2idx["<SOS>"])

    for ch in text:

        if ch in char2idx:
            encoded.append(char2idx[ch])
        else:
            encoded.append(char2idx["<UNK>"])

    encoded.append(char2idx["<EOS>"])

    return encoded
#Decoding function
def decode(tokens):

    result = ""

    for t in tokens:

        if t in idx2char:

            c = idx2char[t]

            if c not in ["<SOS>","<EOS>","<PAD>"]:

                result += c

    return result

#Testing encoder decoder
sample = df.iloc[0]["label"]

print(sample)

encoded = encode(sample)

print(encoded)

decoded = decode(encoded)

print(decoded)
#save vocabulary
import json

with open("char2idx.json","w") as f:

    json.dump(char2idx,f)

with open("idx2char.json","w") as f:

    json.dump(idx2char,f)

print("Vocabulary Saved")



Vocabulary Size: 149
[' ', '!', '"', '%', '(', ')', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '<', '=', '>', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', '\\', ']', '^', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '{', '}', '√', '、', '。', '丙', '个', '为', '乙', '人', '代', '以', '倍', '元', '入', '公', '分', '即', '原', '可', '周', '和', '因', '大', '天', '妈', '将', '小', '岁', '年', '式', '得', '或', '所', '把', '数', '方', '时', '明', '是', '最', '本', '次', '求', '法', '满', '然', '甲', '的', '种', '积', '第', '答', '组', '能', '自', '行', '袋', '解', '该', '负', '足', '面', '页', '项']
Total Tokens: 153
( 2 ) 2 N a O H + C u S O _ { 4 } = N a _ { 2 } S O _ { 4 } + C u ( O H ) _ { 2 } \downarrow
[1, 8, 4, 17, 4, 9, 4, 17, 4, 44, 4, 62, 4, 45, 4, 38, 4, 10, 4, 33, 4, 82, 4, 49, 4, 45, 4, 61, 4, 88, 4, 19, 4, 89, 4, 

In [ ]:
#data processing
#creating pytorch dataset
import os
import torch
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms
#image transform
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((128,512)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])
#Dataset class
class HMEDataset(Dataset):

    def __init__(self, dataframe, image_folder, transform=None):

        self.df = dataframe
        self.image_folder = image_folder
        self.transform = transform

    def __len__(self):

        return len(self.df)

    def __getitem__(self, idx):

        image_name = self.df.iloc[idx]["image"]

        image_path = os.path.join(
            self.image_folder,
            image_name
        )

        image = Image.open(image_path).convert("L")

        if self.transform:
            image = self.transform(image)

        label = self.df.iloc[idx]["label"]

        label = encode(label)

        label = torch.tensor(label)

        return image, label
#create dataset
train_dataset = HMEDataset(
    dataframe=df,
    image_folder="/content/drive/MyDrive/HME100K/train/train_images",
    transform=transform
)

print("Dataset Size:", len(train_dataset))
#Test dataset
image, label = train_dataset[0]

print(image.shape)

print(label)
#PyTorch cannot create a batch from labels of different lengths.
#So we need padding.
#padding function
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):

    images = []

    labels = []

    for img, lab in batch:

        images.append(img)

        labels.append(lab)

    images = torch.stack(images)

    labels = pad_sequence(
        labels,
        batch_first=True,
        padding_value=char2idx["<PAD>"]
    )

    return images, labels
#Dataloader
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    collate_fn=collate_fn
)
#test dataloader
images, labels = next(iter(train_loader))

print(images.shape)

print(labels.shape)

Dataset Size: 74502
torch.Size([1, 128, 512])
tensor([ 1,  8,  4, 17,  4,  9,  4, 17,  4, 44,  4, 62,  4, 45,  4, 38,  4, 10,
         4, 33,  4, 82,  4, 49,  4, 45,  4, 61,  4, 88,  4, 19,  4, 89,  4, 28,
         4, 44,  4, 62,  4, 61,  4, 88,  4, 17,  4, 89,  4, 49,  4, 45,  4, 61,
         4, 88,  4, 19,  4, 89,  4, 10,  4, 33,  4, 82,  4,  8,  4, 45,  4, 38,
         4,  9,  4, 61,  4, 88,  4, 17,  4, 89,  4, 58, 65, 76, 84, 75, 62, 79,
        79, 76, 84,  2])
torch.Size([16, 1, 128, 512])
torch.Size([16, 101])


NameError: name 'lovely' is not defined